# 05 — Sampled link prediction, retrieval, and neighborhood analysis

Three realistic operations are demonstrated on a deterministic fixture: representation retrieval, relation-neighborhood inspection, and a tiny heterogeneous link-prediction smoke. The training result is a software smoke only.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "public":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})

try:
    import torch
    import torch_geometric
except ImportError as exc:
    raise RuntimeError("Install the notebook GNN environment with: uv sync --group notebooks --group gnn") from exc


In [ ]:
from manage_db.public_notebooks import build_sampled_pyg, nearest_embeddings, run_sampled_ml

if MODE != "fixture":
    raise RuntimeError("Train only on an explicitly staged bounded subset with a reviewed leakage policy.")
embedding_uri = Path(KG_ROOT) / "features" / "embeddings" / "text" / "fixture.parquet"
display(nearest_embeddings(embedding_uri, "ENSG00000012048", limit=3))


In [ ]:
edge_uri = Path(KG_ROOT) / "edges" / "disease_associated_gene.parquet"
neighborhood = read_bounded_parquet(edge_uri, limit=100)
display(neighborhood.groupby("x_id").size().rename("sampled_degree").sort_values(ascending=False))


In [ ]:
PYG_ROOT = CACHE / "pyg-ml-sample"
build_sampled_pyg(KG_ROOT, PYG_ROOT, max_nodes_per_type=100, max_edges_per_relation=200)
smoke = run_sampled_ml(PYG_ROOT, seed=13)
print(json.dumps({
    "status": smoke["status"],
    "split_counts": smoke["split_counts"],
    "metrics": smoke["metrics"],
    "validation": smoke["validation"],
}, indent=2))


## Leakage and validity checklist

- Split by biological entity/time/source when the task requires zero-shot or prospective evaluation; a random edge split can leak close duplicates and neighboring evidence.
- Mask treatment/trial outcome text for held-out `molecule_treats_disease` prediction.
- Do not encode labels, split membership, model predictions, or downstream target evidence into inputs.
- Negative samples are unknown/unobserved pairs, not proven biological negatives.
- Smoke loss/accuracy on this fixture is not a benchmark, efficacy estimate, or publication result.
